# Clustering Analysis of Colorado Bird Sightings

This notebook uses clustering analysis on `birdsong_df` to answer three questions:

1. What are the bird hotspots in Colorado during each of the four seasons?
2. How has the prevalence of different species of birds changed over time?
3. What are the typical flight patterns of different bird species?

The notebook also captures dataset snapshots before and after each major transformation, documents model assumptions and tuning choices, and evaluates clustering quality with Silhouette Score and Davies-Bouldin Index.

## Why Clustering Was Chosen

`birdsong_df` is largely unlabeled for the questions we want to answer. We do not have a ground-truth label that says whether a county is a hotspot, which species belong to the same temporal prevalence profile, or which species share similar flight behavior. That makes unsupervised learning the right modeling family.

K-Means is used here because:

- The transformed feature sets are numeric and can be standardized.
- We want interpretable centroids that summarize each cluster.
- The dataset is large enough that a fast centroid-based method is practical.
- We can tune the number of clusters using internal validation metrics.

## Model Assumptions

K-Means assumes:

- Clusters are reasonably compact and separable in feature space.
- Euclidean distance is meaningful after feature scaling.
- Features with larger raw units should not dominate, so scaling is required.
- The chosen number of clusters `k` is not known in advance and must be tuned.

For the flight-pattern analysis, an additional practical assumption is made: because the dataset contains observations rather than true tracked trajectories, flight patterns are approximated using seasonal and monthly geographic movement signatures rather than literal path reconstruction.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='talk')

In [3]:
# Use the in-memory DataFrame when it already exists; otherwise fall back to the saved CSV.
if 'birdsong_df' not in globals():
    birdsong_df = pd.read_csv('..\\data\\birdsong.csv')

birdsong_df = birdsong_df.copy()
birdsong_df.head()

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season
0,Band-tailed Pigeon,2021-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter
1,Pine Warbler,2021-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
2,White-winged Scoter,2021-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter
3,Brown Thrasher,2021-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
4,Bonaparte's Gull,2021-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter


## Snapshot Before Transformations

In [4]:
print(birdsong_df.shape)
birdsong_df.info()
birdsong_df.head(10)

(349430, 14)
<class 'pandas.DataFrame'>
RangeIndex: 349430 entries, 0 to 349429
Data columns (total 14 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   common_name                         349430 non-null  str    
 1   date                                349430 non-null  str    
 2   bird_count                          349430 non-null  float64
 3   county                              349430 non-null  str    
 4   family                              349430 non-null  str    
 5   total_monthly_precipitation         349430 non-null  float64
 6   monthly_max_temp                    349430 non-null  float64
 7   monthly_min_temp                    349430 non-null  float64
 8   urban_population_level              349430 non-null  int64  
 9   county_total_population             349430 non-null  float64
 10  is_fire                             349430 non-null  int64  
 11  is_flood                

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season
0,Band-tailed Pigeon,2021-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter
1,Pine Warbler,2021-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
2,White-winged Scoter,2021-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter
3,Brown Thrasher,2021-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
4,Bonaparte's Gull,2021-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter
5,American Three-toed Woodpecker,2021-01,1.0000,Larimer,Woodpeckers,10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter
6,Greater Roadrunner,2021-01,1.0000,Pueblo,Cuckoos,14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter
7,Graylag x Swan Goose (hybrid),2021-01,3.0000,Denver,"Ducks, Geese, and Waterfowl",6.0800,9.3900,-5.9000,5,"721,246.1760",0,0,0.0000,Winter
8,Black Phoebe,2021-01,1.0000,Pueblo,Tyrant Flycatchers,14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter
9,Northern Mockingbird,2021-01,1.0000,Otero,Mockingbirds and Thrashers,14.8273,7.9229,-7.6471,3,"18,839.5200",0,0,0.0001,Winter


## Core Transformations

The clustering tasks need a consistent temporal format, numeric features, and stable prevalence measures. The major steps below standardize dates, create time-derived features, and build analysis-specific feature tables.

In [5]:
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
}

birdsong_df['date'] = pd.to_datetime(birdsong_df['date'].astype(str))
birdsong_df['year'] = birdsong_df['date'].dt.year
birdsong_df['month'] = birdsong_df['date'].dt.month
birdsong_df['season'] = birdsong_df['month'].map(season_map)
birdsong_df['bird_count'] = pd.to_numeric(birdsong_df['bird_count'], errors='coerce').fillna(1)

if 'bird_count_adjusted_for_population' not in birdsong_df.columns:
    birdsong_df['county_total_population'] = pd.to_numeric(
        birdsong_df.get('county_total_population', np.nan), errors='coerce'
    )
    birdsong_df['bird_count_adjusted_for_population'] = (
        birdsong_df['bird_count'] / birdsong_df['county_total_population']
    )

birdsong_df['bird_count_adjusted_for_population'] = birdsong_df['bird_count_adjusted_for_population'].replace([np.inf, -np.inf], np.nan)
birdsong_df['log_bird_count'] = np.log1p(birdsong_df['bird_count'])

birdsong_df[['common_name', 'date', 'year', 'month', 'season', 'bird_count', 'bird_count_adjusted_for_population', 'log_bird_count']].head(10)

,common_name,date,year,month,season,bird_count,bird_count_adjusted_for_population,log_bird_count
0,Band-tailed Pigeon,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
1,Pine Warbler,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
2,White-winged Scoter,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
3,Brown Thrasher,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
4,Bonaparte's Gull,2021-01-01,2021,1,Winter,2.0000,0.0000,1.0986
5,American Three-toed Woodpecker,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
6,Greater Roadrunner,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
7,Graylag x Swan Goose (hybrid),2021-01-01,2021,1,Winter,3.0000,0.0000,1.3863
8,Black Phoebe,2021-01-01,2021,1,Winter,1.0000,0.0000,0.6931
9,Northern Mockingbird,2021-01-01,2021,1,Winter,1.0000,0.0001,0.6931


### Snapshot After Base Feature Engineering

In [6]:
birdsong_df[['common_name', 'county', 'date', 'season', 'bird_count', 'bird_count_adjusted_for_population', 'monthly_max_temp', 'monthly_min_temp', 'total_monthly_precipitation']].head(10)

,common_name,county,date,season,bird_count,bird_count_adjusted_for_population,monthly_max_temp,monthly_min_temp,total_monthly_precipitation
0,Band-tailed Pigeon,Garfield,2021-01-01,Winter,1.0000,0.0000,1.4122,-10.4589,23.0158
1,Pine Warbler,Boulder,2021-01-01,Winter,1.0000,0.0000,3.8300,-8.2778,9.8737
2,White-winged Scoter,Larimer,2021-01-01,Winter,1.0000,0.0000,1.8541,-10.4572,10.8259
3,Brown Thrasher,Boulder,2021-01-01,Winter,1.0000,0.0000,3.8300,-8.2778,9.8737
4,Bonaparte's Gull,Pueblo,2021-01-01,Winter,2.0000,0.0000,8.5667,-8.4000,14.1278
5,American Three-toed Woodpecker,Larimer,2021-01-01,Winter,1.0000,0.0000,1.8541,-10.4572,10.8259
6,Greater Roadrunner,Pueblo,2021-01-01,Winter,1.0000,0.0000,8.5667,-8.4000,14.1278
7,Graylag x Swan Goose (hybrid),Denver,2021-01-01,Winter,3.0000,0.0000,9.3900,-5.9000,6.0800
8,Black Phoebe,Pueblo,2021-01-01,Winter,1.0000,0.0000,8.5667,-8.4000,14.1278
9,Northern Mockingbird,Otero,2021-01-01,Winter,1.0000,0.0001,7.9229,-7.6471,14.8273


## Shared Utilities

Hyperparameter tuning is handled by testing multiple values of `k` and selecting the value that maximizes Silhouette Score, with Davies-Bouldin Index used as a secondary quality check. This is appropriate because the tasks are unsupervised and do not have true labels.

In [7]:
def evaluate_kmeans_grid(X, k_values=range(2, 9), random_state=42):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        rows.append({
            'k': k,
            'silhouette_score': silhouette_score(X, labels),
            'davies_bouldin_index': davies_bouldin_score(X, labels)
        })
    scores = pd.DataFrame(rows)
    best_k = scores.sort_values(['silhouette_score', 'davies_bouldin_index'], ascending=[False, True]).iloc[0]['k']
    return scores, int(best_k)

def fit_final_kmeans(X, best_k, random_state=42):
    model = KMeans(n_clusters=best_k, random_state=random_state, n_init=20)
    labels = model.fit_predict(X)
    metrics = {
        'silhouette_score': silhouette_score(X, labels),
        'davies_bouldin_index': davies_bouldin_score(X, labels)
    }
    return model, labels, metrics

## 1. Bird Hotspots by Season

A hotspot is modeled here as a county-season combination with similar abundance, diversity, and environmental conditions. County-level aggregation is appropriate because the question asks for regional hotspots rather than individual sightings.

In [8]:
hotspot_features = (
    birdsong_df
    .groupby(['season', 'county'], as_index=False)
    .agg(
        total_bird_count=('bird_count', 'sum'),
        avg_bird_count=('bird_count', 'mean'),
        adjusted_bird_count=('bird_count_adjusted_for_population', 'mean'),
        species_richness=('common_name', 'nunique'),
        avg_precipitation=('total_monthly_precipitation', 'mean'),
        avg_max_temp=('monthly_max_temp', 'mean'),
        avg_min_temp=('monthly_min_temp', 'mean'),
        avg_population=('county_total_population', 'mean'),
        fire_rate=('is_fire', 'mean'),
        flood_rate=('is_flood', 'mean')
    )
)

hotspot_features.head(10)

,season,county,total_bird_count,avg_bird_count,adjusted_bird_count,species_richness,avg_precipitation,avg_max_temp,avg_min_temp,avg_population,fire_rate,flood_rate
0,Autumn,Adams,"10,708.0000",9.3848,0.0000,227,15.7197,22.9850,5.0132,"532,094.3122",0.0000,0.0000
1,Autumn,Alamosa,"2,087.0000",6.4613,0.0004,127,20.5382,20.7467,1.0048,"16,747.5545",0.0000,0.0000
2,Autumn,Arapahoe,"58,368.0000",7.2914,0.0000,292,22.4786,20.7356,4.7098,"673,231.8215",0.0000,0.0000
3,Autumn,Archuleta,"1,342.0000",3.2260,0.0002,129,44.7430,20.2064,4.8084,"13,665.4999",0.0000,0.0000
4,Autumn,Baca,"1,542.0000",2.7340,0.0008,176,21.5782,23.7662,6.0375,"3,583.3067",0.0000,0.0000
5,Autumn,Bent,"8,716.0000",10.8273,0.0019,208,16.4777,25.1373,5.1485,"5,766.4270",0.0000,0.0000
6,Autumn,Boulder,"106,534.0000",10.9558,0.0000,336,22.0492,16.4570,1.7042,"338,342.6022",0.0000,0.0000
7,Autumn,Broomfield,"10,466.0000",4.2736,0.0001,183,20.9944,20.0744,3.3209,"76,304.2841",0.0000,0.0000
8,Autumn,Chaffee,"4,690.0000",5.3054,0.0003,189,25.1782,15.1175,-0.8366,"19,933.0265",0.0000,0.0000
9,Autumn,Cheyenne,"4,750.0000",12.4346,0.0069,112,17.0789,25.7296,7.1963,"1,805.0577",0.0000,0.0000


### Snapshot Before Hotspot Scaling

In [9]:
hotspot_features.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
season,256,4,Autumn,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county,256,64,Adams,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_bird_count,256.0000,NaN,NaN,NaN,"9,640.3828","17,324.5717",64.0000,"1,045.5000","2,793.0000","8,367.5000","106,534.0000"
avg_bird_count,256.0000,NaN,NaN,NaN,11.4188,28.2083,1.5864,3.1456,4.6454,7.8851,316.7348
adjusted_bird_count,256.0000,NaN,NaN,NaN,0.0018,0.0054,0.0000,0.0001,0.0003,0.0013,0.0607
species_richness,256.0000,NaN,NaN,NaN,136.3008,78.7060,9.0000,79.7500,128.0000,181.2500,363.0000
avg_precipitation,256.0000,NaN,NaN,NaN,37.9501,18.4466,5.3338,22.0000,37.4326,51.6490,91.3747
avg_max_temp,256.0000,NaN,NaN,NaN,16.2628,9.1412,-2.4093,9.2158,16.5480,22.9825,33.4594
avg_min_temp,256.0000,NaN,NaN,NaN,0.5752,7.9933,-16.4944,-6.3259,1.0491,5.6284,16.1676
avg_population,256.0000,NaN,NaN,NaN,"92,411.4907","186,857.0988",717.3612,"5,907.2869","15,503.4420","45,341.5386","749,437.1783"


In [10]:
hotspot_numeric_cols = [
    'total_bird_count', 'avg_bird_count', 'adjusted_bird_count', 'species_richness',
    'avg_precipitation', 'avg_max_temp', 'avg_min_temp', 'avg_population',
    'fire_rate', 'flood_rate'
]

hotspot_scaled = hotspot_features.copy()
hotspot_scaled[hotspot_numeric_cols] = StandardScaler().fit_transform(hotspot_scaled[hotspot_numeric_cols])
hotspot_scaled.head(10)

,season,county,total_bird_count,avg_bird_count,adjusted_bird_count,species_richness,avg_precipitation,avg_max_temp,avg_min_temp,avg_population,fire_rate,flood_rate
0,Autumn,Adams,0.0617,-0.0722,-0.3236,1.1546,-1.2075,0.7368,0.5563,2.3577,-0.1357,-0.1896
1,Autumn,Alamosa,-0.4368,-0.1761,-0.2553,-0.1184,-0.9458,0.4915,0.0538,-0.4057,-0.1357,-0.1896
2,Autumn,Arapahoe,2.8181,-0.1466,-0.3249,1.9821,-0.8404,0.4903,0.5183,3.1145,-0.1357,-0.1896
3,Autumn,Archuleta,-0.4799,-0.2910,-0.2831,-0.0929,0.3690,0.4323,0.5306,-0.4222,-0.1357,-0.1896
4,Autumn,Baca,-0.4684,-0.3085,-0.1849,0.5054,-0.8893,0.8225,0.6847,-0.4763,-0.1357,-0.1896
5,Autumn,Bent,-0.0535,-0.0210,0.0231,0.9128,-1.1663,0.9727,0.5733,-0.4646,-0.1357,-0.1896
6,Autumn,Boulder,5.6038,-0.0164,-0.3209,2.5423,-0.8637,0.0213,0.1415,1.3187,-0.1357,-0.1896
7,Autumn,Broomfield,0.0477,-0.2538,-0.3165,0.5945,-0.9210,0.4178,0.3442,-0.0864,-0.1357,-0.1896
8,Autumn,Chaffee,-0.2863,-0.2171,-0.2774,0.6709,-0.6937,-0.1255,-0.1770,-0.3886,-0.1357,-0.1896
9,Autumn,Cheyenne,-0.2828,0.0361,0.9485,-0.3094,-1.1337,1.0377,0.8300,-0.4858,-0.1357,-0.1896


### Snapshot After Hotspot Scaling

In [11]:
hotspot_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
total_bird_count,256.0000,-0.0000,1.0020,-0.5538,-0.4971,-0.3960,-0.0736,5.6038
avg_bird_count,256.0000,0.0000,1.0020,-0.3492,-0.2939,-0.2406,-0.1255,10.8448
adjusted_bird_count,256.0000,-0.0000,1.0020,-0.3261,-0.3102,-0.2642,-0.0855,10.9717
species_richness,256.0000,0.0000,1.0020,-1.6206,-0.7199,-0.1057,0.5722,2.8860
avg_precipitation,256.0000,-0.0000,1.0020,-1.7716,-0.8664,-0.0281,0.7441,2.9018
avg_max_temp,256.0000,0.0000,1.0020,-2.0466,-0.7724,0.0313,0.7365,1.8849
avg_min_temp,256.0000,-0.0000,1.0020,-2.1397,-0.8651,0.0594,0.6334,1.9545
avg_population,256.0000,-0.0000,1.0020,-0.4917,-0.4638,-0.4124,-0.2524,3.5231
fire_rate,256.0000,-0.0000,1.0020,-0.1357,-0.1357,-0.1357,-0.1357,9.5926
flood_rate,256.0000,0.0000,1.0020,-0.1896,-0.1896,-0.1896,-0.1896,11.6543


In [12]:
season_hotspot_results = []
season_hotspot_metrics = []

for season_name, season_df in hotspot_scaled.groupby('season'):
    X = season_df[hotspot_numeric_cols]
    scores, best_k = evaluate_kmeans_grid(X, k_values=range(2, min(8, len(season_df) - 1) + 1))
    model, labels, metrics = fit_final_kmeans(X, best_k)

    clustered = season_df.copy()
    clustered['cluster'] = labels
    centroids = clustered.groupby('cluster')[hotspot_numeric_cols].mean()
    hotspot_cluster = centroids[['total_bird_count', 'species_richness', 'adjusted_bird_count']].mean(axis=1).idxmax()
    hotspots = clustered[clustered['cluster'] == hotspot_cluster].copy()
    hotspots = hotspots.sort_values(['total_bird_count', 'species_richness'], ascending=False)

    metrics_row = {
        'season': season_name,
        'best_k': best_k,
        'silhouette_score': metrics['silhouette_score'],
        'davies_bouldin_index': metrics['davies_bouldin_index']
    }

    season_hotspot_results.append(hotspots)
    season_hotspot_metrics.append(metrics_row)

season_hotspot_metrics_df = pd.DataFrame(season_hotspot_metrics).sort_values('season')
season_hotspot_metrics_df

,season,best_k,silhouette_score,davies_bouldin_index
0,Autumn,2,0.6125,0.7601
1,Spring,3,0.4686,0.6376
2,Summer,3,0.5748,0.7549
3,Winter,2,0.7216,0.5922


In [13]:
seasonal_hotspot_summary = (
    pd.concat(season_hotspot_results, ignore_index=True)
    [['season', 'county', 'cluster', 'total_bird_count', 'species_richness', 'adjusted_bird_count']]
    .sort_values(['season', 'total_bird_count', 'species_richness'], ascending=[True, False, False])
)

seasonal_hotspot_summary.groupby('season').head(5)

,season,county,cluster,total_bird_count,species_richness,adjusted_bird_count
0,Autumn,Boulder,1,5.6038,2.5423,-0.3209
1,Autumn,Logan,1,5.3232,1.0146,0.2487
2,Autumn,Denver,1,4.1438,1.0401,-0.3204
3,Autumn,Larimer,1,2.8401,2.6696,-0.3236
4,Autumn,Arapahoe,1,2.8181,1.9821,-0.3249
9,Spring,Conejos,2,0.9997,-0.8950,4.5982
10,Summer,Boulder,2,1.5977,1.9439,-0.3252
11,Summer,Larimer,2,1.2621,2.0203,-0.3253
12,Summer,Jefferson,2,0.7381,1.6129,-0.3261
13,Winter,Kit Carson,1,2.7580,-0.9586,7.6819


The table above answers the seasonal hotspot question by identifying the counties that fall into the highest-abundance and highest-diversity cluster for each season.

## 2. Species Prevalence Changes Over Time

To study how species prevalence changes over time, each species is represented by a monthly prevalence profile. Prevalence is defined as total observed count per month, and a small `log1p` transform is applied to reduce the influence of unusually large counts.

In [14]:
species_monthly = (
    birdsong_df
    .groupby(['common_name', 'date'], as_index=False)
    .agg(monthly_bird_count=('bird_count', 'sum'))
)
species_monthly['log_monthly_bird_count'] = np.log1p(species_monthly['monthly_bird_count'])
species_monthly.head(10)

,common_name,date,monthly_bird_count,log_monthly_bird_count
0,Acadian Flycatcher,2023-05-01,1.0000,0.6931
1,Acadian Flycatcher,2024-05-01,1.0000,0.6931
2,Acadian Flycatcher,2024-06-01,1.0000,0.6931
3,Acorn Woodpecker,2021-01-01,9.0000,2.3026
4,Acorn Woodpecker,2021-02-01,6.0000,1.9459
5,Acorn Woodpecker,2021-03-01,2.0000,1.0986
6,Acorn Woodpecker,2021-05-01,3.0000,1.3863
7,Acorn Woodpecker,2021-06-01,16.0000,2.8332
8,Acorn Woodpecker,2021-07-01,15.0000,2.7726
9,Acorn Woodpecker,2021-08-01,18.0000,2.9444


### Snapshot Before Prevalence Pivot

In [15]:
species_monthly.head(10)

,common_name,date,monthly_bird_count,log_monthly_bird_count
0,Acadian Flycatcher,2023-05-01,1.0000,0.6931
1,Acadian Flycatcher,2024-05-01,1.0000,0.6931
2,Acadian Flycatcher,2024-06-01,1.0000,0.6931
3,Acorn Woodpecker,2021-01-01,9.0000,2.3026
4,Acorn Woodpecker,2021-02-01,6.0000,1.9459
5,Acorn Woodpecker,2021-03-01,2.0000,1.0986
6,Acorn Woodpecker,2021-05-01,3.0000,1.3863
7,Acorn Woodpecker,2021-06-01,16.0000,2.8332
8,Acorn Woodpecker,2021-07-01,15.0000,2.7726
9,Acorn Woodpecker,2021-08-01,18.0000,2.9444


In [16]:
species_prevalence_wide = (
    species_monthly
    .pivot(index='common_name', columns='date', values='log_monthly_bird_count')
    .fillna(0)
)

species_prevalence_scaled = pd.DataFrame(
    StandardScaler().fit_transform(species_prevalence_wide),
    index=species_prevalence_wide.index,
    columns=species_prevalence_wide.columns
)

species_prevalence_scaled.iloc[:10, :8]

date,2021-01-01,2021-02-01,2021-03-01,2021-04-01,2021-05-01,2021-06-01,2021-07-01,2021-08-01
common_name,,,,,,,,
Acadian Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.9999,-0.9614,-0.9789
Acorn Woodpecker,0.3430,0.2323,-0.2563,-0.9609,-0.4769,0.4286,0.3719,0.4025
African Collared-Dove,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.6504,-0.9614,-0.9789
Alder Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-0.0017,-0.9999,-0.9614,-0.4635
American Avocet,-0.7252,-0.7075,0.8372,1.6491,1.2740,1.7376,1.6701,1.6764
American Barn Owl,0.3872,0.1579,-0.0137,0.3441,0.4303,0.3307,0.0386,0.2916
American Bittern,-0.7252,-0.7075,-0.7781,0.5706,0.6175,0.8069,0.4285,0.2592
American Coot,2.4899,2.5865,1.9732,2.1877,1.7991,1.4386,1.6099,1.8494
American Crow,1.5931,1.8658,2.0872,1.3299,0.8730,1.1048,1.9173,3.1799


### Snapshot After Prevalence Scaling

In [17]:
species_prevalence_scaled.describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
date,,,,,,,,
2021-01-01,567.0000,0.0000,1.0009,-0.7252,-0.7252,-0.7252,0.8369,3.8976
2021-02-01,567.0000,0.0000,1.0009,-0.7075,-0.7075,-0.7075,0.8064,3.4005
2021-03-01,567.0000,-0.0000,1.0009,-0.7781,-0.7781,-0.7781,0.8450,3.6693
2021-04-01,567.0000,0.0000,1.0009,-0.9609,-0.9609,-0.2928,0.8516,2.9647
2021-05-01,567.0000,0.0000,1.0009,-1.1959,-1.1959,0.2421,0.8119,2.3379
2021-06-01,567.0000,0.0000,1.0009,-0.9999,-0.9999,-0.3009,0.9413,2.6745
2021-07-01,567.0000,0.0000,1.0009,-0.9614,-0.9614,-0.2947,0.9387,2.3589
2021-08-01,567.0000,0.0000,1.0009,-0.9789,-0.9789,-0.2238,0.8794,3.1799
2021-09-01,567.0000,0.0000,1.0009,-1.0026,-1.0026,-0.2355,0.8222,3.6703


In [18]:
prevalence_scores, prevalence_best_k = evaluate_kmeans_grid(species_prevalence_scaled, k_values=range(2, 9))
prevalence_model, prevalence_labels, prevalence_metrics = fit_final_kmeans(species_prevalence_scaled, prevalence_best_k)

species_prevalence_clusters = species_prevalence_scaled.copy()
species_prevalence_clusters['cluster'] = prevalence_labels
species_prevalence_clusters['avg_scaled_prevalence'] = species_prevalence_scaled.mean(axis=1)
species_prevalence_clusters['prevalence_trend'] = (
    species_prevalence_wide.iloc[:, -12:].mean(axis=1) - species_prevalence_wide.iloc[:, :12].mean(axis=1)
)

prevalence_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.4960,0.7972
1,3,0.4808,0.8617
2,4,0.4566,0.9542
3,5,0.4583,0.9241
4,6,0.4310,1.0360
5,7,0.4067,1.1874
6,8,0.3941,1.2088


In [19]:
prevalence_cluster_summary = (
    species_prevalence_clusters
    .groupby('cluster')
    .agg(
        species_count=('avg_scaled_prevalence', 'size'),
        mean_scaled_prevalence=('avg_scaled_prevalence', 'mean'),
        mean_trend=('prevalence_trend', 'mean')
    )
    .sort_values('mean_trend', ascending=False)
)

species_prevalence_clusters[['cluster', 'avg_scaled_prevalence', 'prevalence_trend']].sort_values('prevalence_trend', ascending=False).head(15)

date,cluster,avg_scaled_prevalence,prevalence_trend
common_name,,,
Cassia Crossbill,0,-0.5162,0.9619
Gambel's Quail,1,1.6187,0.9089
Greater Roadrunner,1,0.1971,0.8516
Yellow-billed Loon,0,-0.4781,0.8496
Northern Parula,0,-0.2312,0.7143
Sagebrush Sparrow,1,0.3218,0.7015
American Barn Owl,1,0.3149,0.7004
Muscovy Duck,0,-0.4903,0.6880
Anhinga,0,-0.8338,0.6816


The prevalence clusters separate species into shared temporal patterns such as increasing prevalence, declining prevalence, and relatively stable seasonal occurrence. Species with the largest positive `prevalence_trend` have become more prevalent in the later part of the time series, while strongly negative values indicate decline.

## 3. Typical Flight Patterns by Species

True flight paths are not directly recorded in `birdsong_df`, so this section estimates flight-pattern types using geographic movement signatures. For each species, the model summarizes where observations are concentrated throughout the year and how much the center of activity shifts across months.

In [20]:
raw_bird_data = pd.read_csv('..\\data\\ebird_co.csv')
raw_bird_data['obsDt'] = pd.to_datetime(raw_bird_data['obsDt'], errors='coerce')
raw_bird_data['queryDate'] = pd.to_datetime(raw_bird_data['queryDate'], errors='coerce')
raw_bird_data['effective_date'] = raw_bird_data['obsDt'].fillna(raw_bird_data['queryDate'])
raw_bird_data['month'] = raw_bird_data['effective_date'].dt.month
raw_bird_data['howMany'] = pd.to_numeric(raw_bird_data['howMany'], errors='coerce').fillna(1)

species_monthly_geo = (
    raw_bird_data
    .dropna(subset=['comName', 'lat', 'lng', 'effective_date'])
    .groupby(['comName', 'month'], as_index=False)
    .agg(
        mean_lat=('lat', 'mean'),
        mean_lng=('lng', 'mean'),
        lat_std=('lat', 'std'),
        lng_std=('lng', 'std'),
        mean_count=('howMany', 'mean'),
        total_count=('howMany', 'sum')
    )
)

species_monthly_geo.head(10)

,comName,month,mean_lat,mean_lng,lat_std,lng_std,mean_count,total_count
0,Acadian Flycatcher,5,38.7743,-103.7272,0.3375,1.0347,1.0000,2.0000
1,Acadian Flycatcher,6,38.4868,-104.4380,NaN,NaN,1.0000,1.0000
2,Acorn Woodpecker,1,37.2648,-107.9453,0.0022,0.0074,2.4667,37.0000
3,Acorn Woodpecker,2,37.2696,-107.9289,0.0140,0.0480,3.0909,34.0000
4,Acorn Woodpecker,3,37.2887,-107.8637,0.0329,0.1127,1.5000,3.0000
5,Acorn Woodpecker,4,37.2622,-107.8949,0.0357,0.0756,1.3333,8.0000
6,Acorn Woodpecker,5,37.5293,-107.7564,0.8789,0.7399,1.4286,20.0000
7,Acorn Woodpecker,6,37.8502,-106.7648,0.7548,1.5121,2.0690,60.0000
8,Acorn Woodpecker,7,37.9895,-106.4914,0.7764,1.5598,2.0278,73.0000
9,Acorn Woodpecker,8,37.3599,-107.7807,0.4149,0.7919,2.6667,48.0000


### Snapshot Before Flight-Pattern Feature Engineering

In [21]:
species_monthly_geo.head(10)

,comName,month,mean_lat,mean_lng,lat_std,lng_std,mean_count,total_count
0,Acadian Flycatcher,5,38.7743,-103.7272,0.3375,1.0347,1.0000,2.0000
1,Acadian Flycatcher,6,38.4868,-104.4380,NaN,NaN,1.0000,1.0000
2,Acorn Woodpecker,1,37.2648,-107.9453,0.0022,0.0074,2.4667,37.0000
3,Acorn Woodpecker,2,37.2696,-107.9289,0.0140,0.0480,3.0909,34.0000
4,Acorn Woodpecker,3,37.2887,-107.8637,0.0329,0.1127,1.5000,3.0000
5,Acorn Woodpecker,4,37.2622,-107.8949,0.0357,0.0756,1.3333,8.0000
6,Acorn Woodpecker,5,37.5293,-107.7564,0.8789,0.7399,1.4286,20.0000
7,Acorn Woodpecker,6,37.8502,-106.7648,0.7548,1.5121,2.0690,60.0000
8,Acorn Woodpecker,7,37.9895,-106.4914,0.7764,1.5598,2.0278,73.0000
9,Acorn Woodpecker,8,37.3599,-107.7807,0.4149,0.7919,2.6667,48.0000


In [22]:
flight_pattern_features = (
    species_monthly_geo
    .groupby('comName', as_index=False)
    .agg(
        annual_mean_lat=('mean_lat', 'mean'),
        annual_mean_lng=('mean_lng', 'mean'),
        monthly_lat_shift=('mean_lat', lambda x: x.max() - x.min()),
        monthly_lng_shift=('mean_lng', lambda x: x.max() - x.min()),
        avg_lat_spread=('lat_std', 'mean'),
        avg_lng_spread=('lng_std', 'mean'),
        avg_monthly_count=('mean_count', 'mean'),
        total_observed_count=('total_count', 'sum'),
        active_months=('month', 'nunique')
    )
)

flight_pattern_features['log_total_observed_count'] = np.log1p(flight_pattern_features['total_observed_count'])
flight_pattern_features = flight_pattern_features.drop(columns='total_observed_count')
flight_pattern_features.head(10)

,comName,annual_mean_lat,annual_mean_lng,monthly_lat_shift,monthly_lng_shift,avg_lat_spread,avg_lng_spread,avg_monthly_count,active_months,log_total_observed_count
0,Acadian Flycatcher,38.6306,-104.0826,0.2875,0.7108,0.3375,1.0347,1.0000,2,1.3863
1,Acorn Woodpecker,37.4068,-107.6829,0.7272,1.4540,0.2449,0.4191,2.3243,12,5.9135
2,African Collared-Dove,39.6239,-105.5351,1.0844,4.3772,0.1509,0.3303,1.0278,9,3.0445
3,Alder Flycatcher,39.1897,-103.6650,0.6264,1.3641,0.9549,1.2095,1.0000,4,3.7377
4,American Avocet,39.5182,-105.5852,0.7088,3.7837,0.7975,1.1437,6.7204,10,9.0990
5,American Barn Owl,39.6607,-104.7494,0.4634,0.7441,0.8686,1.1441,1.3276,12,6.9058
6,American Bittern,38.8727,-105.6623,1.0368,2.6214,1.0952,0.7632,1.2204,8,6.4216
7,American Coot,39.3812,-105.4794,0.7044,0.5597,0.9041,1.2101,20.3854,12,10.5178
8,American Crow,39.6302,-105.3672,0.1196,0.3983,0.5375,0.7670,70.7691,12,11.7723
9,American Dipper,39.2335,-106.2227,0.5051,0.4467,1.0616,1.0317,1.4226,12,7.8403


### Snapshot After Flight-Pattern Feature Engineering

In [23]:
flight_numeric_cols = [
    'annual_mean_lat', 'annual_mean_lng', 'monthly_lat_shift', 'monthly_lng_shift',
    'avg_lat_spread', 'avg_lng_spread', 'avg_monthly_count', 'active_months',
    'log_total_observed_count'
]

flight_pattern_scaled = flight_pattern_features.copy()
flight_pattern_scaled[flight_numeric_cols] = StandardScaler().fit_transform(flight_pattern_scaled[flight_numeric_cols])
flight_pattern_scaled.head(10)

,comName,annual_mean_lat,annual_mean_lng,monthly_lat_shift,monthly_lng_shift,avg_lat_spread,avg_lng_spread,avg_monthly_count,active_months,log_total_observed_count
0,Acadian Flycatcher,-1.0169,1.0779,-0.9084,-0.5588,-1.2794,0.2103,-0.2603,-1.5013,-1.5489
1,Acorn Woodpecker,-3.0346,-2.3939,-0.3644,0.0324,-1.6041,-1.0413,-0.1252,1.0375,0.0549
2,African Collared-Dove,0.6208,-0.3228,0.0775,2.3579,-1.9341,-1.2219,-0.2575,0.2758,-0.9614
3,Alder Flycatcher,-0.0951,1.4807,-0.4891,-0.0391,0.8879,0.5656,-0.2603,-0.9936,-0.7159
4,American Avocet,0.4464,-0.3710,-0.3872,1.8858,0.3352,0.4319,0.3234,0.5297,1.1834
5,American Barn Owl,0.6815,0.4350,-0.6907,-0.5324,0.5850,0.4326,-0.2269,1.0375,0.4065
6,American Bittern,-0.6177,-0.4453,0.0186,0.9612,1.3801,-0.3417,-0.2378,0.0219,0.2350
7,American Coot,0.2207,-0.2690,-0.3927,-0.6790,0.7094,0.5669,1.7179,1.0375,1.6861
8,American Crow,0.6312,-0.1608,-1.1161,-0.8075,-0.5774,-0.3339,6.8596,1.0375,2.1305
9,American Dipper,-0.0228,-0.9858,-0.6392,-0.7689,1.2624,0.2042,-0.2172,1.0375,0.7376


In [24]:
flight_scores, flight_best_k = evaluate_kmeans_grid(flight_pattern_scaled[flight_numeric_cols], k_values=range(2, 9))
flight_model, flight_labels, flight_metrics = fit_final_kmeans(flight_pattern_scaled[flight_numeric_cols], flight_best_k)

flight_pattern_scaled['cluster'] = flight_labels
flight_scores

ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
flight_cluster_summary = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .groupby('cluster')
    .agg(
        species_count=('comName', 'size'),
        monthly_lat_shift=('monthly_lat_shift', 'mean'),
        monthly_lng_shift=('monthly_lng_shift', 'mean'),
        active_months=('active_months', 'mean'),
        avg_monthly_count=('avg_monthly_count', 'mean')
    )
    .sort_values(['monthly_lat_shift', 'monthly_lng_shift'], ascending=False)
)

flight_cluster_summary

Clusters with larger monthly latitude and longitude shifts represent stronger migration-like movement patterns, while clusters with low geographic shift and many active months represent more resident or locally stable species.

## Model Evaluation

In [ ]:
evaluation_summary = pd.DataFrame([
    {
        'analysis': 'Seasonal hotspots',
        'best_k': ', '.join(f"{row.season}:{row.best_k}" for row in season_hotspot_metrics_df.itertuples()),
        'silhouette_score': season_hotspot_metrics_df['silhouette_score'].mean(),
        'davies_bouldin_index': season_hotspot_metrics_df['davies_bouldin_index'].mean()
    },
    {
        'analysis': 'Species prevalence over time',
        'best_k': prevalence_best_k,
        'silhouette_score': prevalence_metrics['silhouette_score'],
        'davies_bouldin_index': prevalence_metrics['davies_bouldin_index']
    },
    {
        'analysis': 'Flight-pattern types',
        'best_k': flight_best_k,
        'silhouette_score': flight_metrics['silhouette_score'],
        'davies_bouldin_index': flight_metrics['davies_bouldin_index']
    }
])

evaluation_summary

## Challenges and Solutions

- **Challenge:** Bird counts span very different scales across counties and species.  
  **Solution:** Standardization and `log1p` transforms were used to keep large counts from dominating Euclidean distance.

- **Challenge:** The hotspot question is geographic, but `birdsong_df` is observation-level rather than county-season labeled.  
  **Solution:** The data was aggregated to county-season summaries that preserve abundance, diversity, and environmental context.

- **Challenge:** Flight paths are not directly observed.  
  **Solution:** Flight behavior was approximated with monthly geographic shifts derived from the raw eBird latitude and longitude fields.

- **Challenge:** Choosing `k` is subjective in unsupervised learning.  
  **Solution:** Multiple values of `k` were compared using both Silhouette Score and Davies-Bouldin Index.

## Final Interpretation Notes

- Higher Silhouette Score is better because it indicates tighter, better-separated clusters.
- Lower Davies-Bouldin Index is better because it indicates lower within-cluster dispersion relative to between-cluster separation.
- If either metric is weak for a task, the clusters may still be useful for exploration, but they should be interpreted as soft groupings rather than definitive ecological classes.